#  OLR-Based Index of the MaddenJulian-Oscillation (OMI)


```{image} ../thumbnails/thumbnail.png
:alt: Project Pythia logo
:width: 200px
```

In [15]:
#reference for the location plot 
#Donald, Alexis & Meinke, Holger & Power, Brendan & Wheeler, Matthew & Ribbe, Joachim. (2004). Forecasting with the Madden-Julian Oscillation and the applications for risk management. 

## Overview

The **Madden–Julian Oscillation (MJO)** is an eastward-propagating tropical disturbance characterized by large-scale anomalies in atmospheric circulation and deep convection.

Its state is commonly described using the **Real-time Multivariate MJO (RMM) index**, which is derived from the leading two multivariate empirical orthogonal functions (EOFs) of equatorially averaged outgoing longwave radiation (OLR) and zonal winds at 200 hPa and 850 hPa, following Wheeler and Hendon RMM Index (Wheeler and Hendon, 2004). Although the RMM index includes OLR in its formulation, it tends to be more strongly influenced by large-scale circulation anomalies than by convection itself. In contrast, another index, the **OLR-based MJO Index (OMI)** is constructed more directly from convection, using the leading two daily-varying EOFs of OLR.

Both the RMM index and OMI defines eight phases (1–8) that represent the canonical eastward progression of MJO-related convection and circulation anomalies around the tropics. Phases 1–4 correspond primarily to enhanced convection over the Indian Ocean, while phases 5–8 indicate active MJO conditions over the western Pacific (Kiladis et al., 2014).

```{image} figures/omi-phase-daigram-example.png
:alt: sample phase daigram
:width: 200px
```

```{image} figures/Approximate-locations-of-the-MJO.jpg
:alt: location of MJO
:width: 200px
```

## Prerequisites

### Data Access

1. RMM index data were obtained from the Australian Bureau of Meteorology: [Link](http://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt).
1. OMI data were obtained from NOAA’s Earth System Research Laboratory (ESRL): [Link](https://www.esrl.noaa.gov/psd/mjo/mjoindex/omi.1x.txt).
1. global 3-D field (OLR, U850, U200, geopotential height,..) can be downloaded using the [notebook](spectral-analysis-cookbook/notebooks/SpectralAnalysisDataSetAccess.ipynb)

Or if you want to compute these indices with your data:

1. Follow the tutuorial for RMM index (**Link the repo**)
1. Follow the package from [Hoffmann et al, 2021](https://psl.noaa.gov/mjo/mjoindex/pdf/hoffmann21.pdf) - [see their software](https://doi.org/10.5281/zenodo.3957857) and [see their repo](https://github.com/cghoffmann/mjoindices)

## Workflow

1. Load the two indices
1. Compare and plot differences between the indices
1. Regression onto the Principal Components
    - Regress each global 3-D field (OLR, U850, U200, geopotential height,...) on RMM1 and RMM2.
    - Reconstruct the MJO life cycle by taking linear combinations of the two regression maps: -PC2, (PC1 - PC2)/sqrt(2), PC1, (PC1 + PC2)/sqrt(2), PC2.
    - Compare the reconstructed maps with Wheeler and Hendon (2004) and Adames and Wallace (2014).

## Imports

In [7]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import s3fs
import xarray as xr


In [4]:
# hide-cell
def load_mjo(url, kind="rmm"):
    """
    Unified loader for MJO indices: RMM or OMI.

    Returns standardized dataframe:
    index = datetime
    columns = pc1, pc2, phase, amplitude
    """

    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    data = StringIO(r.text)

    if kind.lower() == "rmm":

        df = pd.read_csv(
            data,
            sep=r"\s+",
            header=None,
            skiprows=2,
            engine="python"
        )

        df = df.iloc[:, :7]
        df.columns = ["year", "month", "day", "pc1", "pc2", "phase", "amplitude"]

    elif kind.lower() == "omi":

        df = pd.read_csv(
            data,
            sep=r"\s+",
            header=None,
            comment="#",
            engine="python"
        )

        df = df.iloc[:, :7]
        df.columns = ["year", "month", "day", "hour", "pc1", "pc2", "amplitude"]

    else:
        raise ValueError("kind must be 'rmm' or 'omi'")

    # datetime index
    df["date"] = pd.to_datetime(df[["year", "month", "day"]])
    df = df.set_index("date")

    # --- OMI: compute phase if needed ---
    if kind.lower() == "omi":
        angle = np.arctan2(df["pc2"].values, df["pc1"].values)
        df["phase"] = ((np.degrees(angle) + 360) % 360 // 45 + 1).astype(int)

    # select standard output
    return df[["pc1", "pc2", "phase", "amplitude"]]


## Getting starting

Lets read the data get started

In [5]:
url_rmm = "https://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt"
url_omi = "https://www.esrl.noaa.gov/psd/mjo/mjoindex/omi.1x.txt"

rmm = load_mjo(url_rmm, kind="rmm")
omi = load_mjo(url_omi, kind="omi")

#rmm.head()
#omi.head()

In [6]:
print("RMM period:")
print(rmm.index.min(), "to", rmm.index.max())

print("\nOMI period:")
print(omi.index.min(), "to", omi.index.max())


RMM period:
1974-06-01 00:00:00 to 2024-02-24 00:00:00

OMI period:
1979-01-01 00:00:00 to 2024-05-20 00:00:00


Now lets get the 3D fields to perform the regression

In [8]:
URL = 'https://js2.jetstream-cloud.org:8001/' #Locate and read a file
fs = s3fs.S3FileSystem(anon=True, client_kwargs=dict(endpoint_url=URL))

olr_noaa_store = s3fs.S3Map(
    root=f'pythia/olr_noaa.zarr',
    s3=fs,
    check=False
)

# Open with xarray
olr_noaa = xr.open_zarr(olr_noaa_store)
olr_noaa


<xarray.Dataset> Size: 242MB
Dimensions:                        (time: 16802, lat: 25, lon: 144)
Coordinates:
  * lat                            (lat) float32 100B 30.0 27.5 ... -27.5 -30.0
  * lon                            (lon) float32 576B 0.0 2.5 ... 355.0 357.5
  * time                           (time) datetime64[ns] 134kB 1979-01-01T12:...
Data variables:
    __xarray_dataarray_variable__  (time, lat, lon) float32 242MB dask.array<chunksize=(1024, 25, 144), meta=np.ndarray>

Now lets save the required variables on a single variable for easy data handling

## Comparison of RMM versus ONI
Now that we have the dataset all sorted, let us with comparing the two indices.
For that we can easily plot a timeseries plot of both pc1 and pc2 components from the two methods.


In [ ]:
# as well as any and all of your code cells
print("Hello world!")

### A content subsection
Divide and conquer your objectives with Markdown subsections, which will populate the helpful navbar in Jupyter Lab and here on the Jupyter Book!

In [ ]:
# some subsection code
a = [1, 2, 3, 4, 5]
[i + 2 for i in a]

### Another content subsection
Keep up the good work! A note, *try to avoid using code comments as narrative*, and instead let them only exist as brief clarifications where necessary.

## Your second content section
Here we can move on to our second objective, and we can demonstrate...

### A subsection to the second section

#### a quick demonstration

##### of further and further

###### header levels

as well as $m = a * t / h$ text! Similarly, you have access to other $\LaTeX$ equation [**functionality**](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Typesetting%20Equations.html) via MathJax:

\begin{align}
\dot{x} & = \sigma(y-x) \\
\dot{y} & = \rho x - y - xz \\
\dot{z} & = -\beta z + xy
\end{align}

Check out [**any number of helpful Markdown resources**](https://www.markdownguide.org/basic-syntax/) for further customizing your notebooks and the [**MyST Syntax Overview**](https://mystmd.org/guide/syntax-overview) for MyST-specific formatting information. Don't hesitate to ask questions if you have problems getting it to look *just right*.

## Last Section

You can add [admonitions using MyST syntax](https://mystmd.org/guide/admonitions):

:::{note}
Your relevant information here!
:::

Some other admonitions you can put in ([there are 10 total](https://mystmd.org/guide/admonitions#admonitions-list)):

:::{hint}
A helpful hint.
:::

:::{warning}
Be careful!
:::

:::{danger}
Scary stuff be here.
:::

We also suggest checking out MyST's [brief demonstration](https://mystmd.org/guide/notebook-configuration#notebook-cell-tags) on adding cell tags to your cells in Jupyter Notebook, Lab, or manually. See [this table](https://mystmd.org/guide/notebook-configuration#tbl-notebook-cell-tags) for a list of supported cell tags, which you can use to customize how your code content is displayed and even [demonstrate errors](https://mystmd.org/guide/execute-notebooks#allow-a-code-cell-to-error-without-failing-the-build) without altogether crashing our loyal army of machines!

---

## Summary
Add one final `---` marking the end of your body of content, and then conclude with a brief single paragraph summarizing at a high level the key pieces that were learned and how they tied to your objectives. Look to reiterate what the most important takeaways were.

### What's next?
Let Jupyter book tie this to the next (sequential) piece of content that people could move on to down below and in the sidebar. However, if this page uniquely enables your reader to tackle other nonsequential concepts throughout this book, or even external content, link to it here!

## Resources and references
Finally, be rigorous in your citations and references as necessary. Give credit where credit is due. Also, feel free to link to relevant external material, further reading, documentation, etc. Then you're done! Give yourself a quick review, a high five, and send us a pull request. A few final notes:
 - `Kernel > Restart Kernel and Run All Cells...` to confirm that your notebook will cleanly run from start to finish
 - `Kernel > Restart Kernel and Clear All Outputs...` before committing your notebook, our machines will do the heavy lifting
 - Take credit! Provide author contact information if you'd like; if so, consider adding information here at the bottom of your notebook
 - Give credit! Attribute appropriate authorship for referenced code, information, images, etc.
 - Only include what you're legally allowed: **no copyright infringement or plagiarism**
 
Thank you for your contribution!